# Simulator tour
A hands-on tour of the Simulator stage (`galactic_tidal_constraints.simr.sim`): it runs one SPAM galaxy
collision and returns arrays. Run the cells top to bottom.

**To run it** (from this repo's root, in bash or PowerShell), then open the `127.0.0.1` link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
- `-v "${PWD}:/gtc"` mounts this repo into the container, so edits on Windows show up immediately.
- `UV_PROJECT_ENVIRONMENT=/tmp/venv` keeps the Linux virtual environment inside the container, out of the repo.
  It's rebuilt each time the container starts (a minute or two, mostly compiling SPAM).
- Our package is installed in editable mode. After editing a `.py` file, restart the kernel to pick it up.

In [ ]:
%matplotlib inline
from dataclasses import asdict, replace

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import sim

## The 22 collision parameters
A SPAM model is 22 numbers, usually written as a comma-separated model string. In code, a model is a plain
array of shape `(22,)`, and `sim.PARAM_NAMES` names each position, using the names in SPAM's Fortran:

| Names | Meaning |
|---|---|
| `x, y, z, vx, vy, vz` | The secondary's position and velocity relative to the primary |
| `mass1, mass2` | The two masses |
| `rout1, rout2` | The two disk radii |
| `phi1, phi2, theta1, theta2` | The two disks' orientations, in degrees |
| `epsilon1, epsilon2` | Softening for gravity between the two centers |
| `rscale1_1` … `rscale2_3` | Three scale values per galaxy (usually all 0) |

Galaxy 1 is the primary, and stays at the origin. Galaxy 2 is the secondary. This model is SDSS
587722984435351614, from JSPAM's `archive/data.txt`. Its line has 34 values; like SPAM,
`parse_model_string` reads the first 22.

In [ ]:
MODEL_STRING = "-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0"

params = sim.parse_model_string(MODEL_STRING)
dict(zip(sim.PARAM_NAMES, params.tolist()))

To read or change one value by name, look up its position in `sim.PARAM_INDEX`. Copy the array first, so the
original model stays as it was.

In [ ]:
heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2
print("mass2:", params[sim.PARAM_INDEX["mass2"]], "->", heavier[sim.PARAM_INDEX["mass2"]])

## How to simulate: `SimulatorConfig`
Everything that sets a run's result, apart from the 22 parameters and the seed, is one `sim.SimulatorConfig`:

| Field | Default | Meaning |
|---|---|---|
| `n1`, `n2` | 1000, 1000 | Particles in the primary's and secondary's disks |
| `heat1`, `heat2` | 0.0, 0.0 | Random motion added to each disk's particle velocities |
| `heat_mode` | `"centered"` | How heat is added (see Disk heat below) |
| `lnl`, `r_scale`, `m_halo`, `m_bulge`, `h_bulge`, `h_disk` | Wallin's: 0.001, 10, 5.8, 0.3333, 0.2, 1.0 | The unit galaxy's mass model (see below) |
| `normalization` | `"total"` | What the unit galaxy is divided by (see below) |

It can't be changed after it's made (to try a different value, make a changed copy with `replace`), and it
checks its values when it's made. `asdict` gives every field, defaults included.

In [ ]:
config = sim.SimulatorConfig(n1=2000, n2=2000)
print(config)
print(asdict(config))
print(replace(config, heat1=0.5))

## One SPAM run
`sim.simulate(params, config, seed)` runs the collision and returns x, y, z, vx, vy, vz of every particle at the
end, shape `(n1 + n2 + 1, 6)`. The last row is the secondary's center. Runs always use SPAM's extended potential
(halo, disk and bulge).

In [ ]:
N1, N2 = config.n1, config.n2

particles = sim.simulate(params, config, seed=1)
print("shape:", particles.shape)
print("secondary's center (x, y, z, vx, vy, vz):", particles[-1])

The secondary's center nearly matches `x, y, z, vx, vy, vz` above. The 22 values describe the system *now*,
as observed. SPAM first runs the two centers backward to find a start time before the encounter, places the
disks there, then runs everything forward and stops at about "now" (a step or two past it). So a run ends at
the observed moment, with the tidal features formed.

Split the rows into the two disks and plot x against y:

In [ ]:
primary = particles[:N1]
secondary = particles[N1:N1 + N2]
center2 = particles[-1]

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(primary[:, 0], primary[:, 1], s=1, label="primary disk")
ax.scatter(secondary[:, 0], secondary[:, 1], s=1, label="secondary disk")
ax.plot(0, 0, "k+", ms=12, label="primary center")
ax.plot(center2[0], center2[1], "kx", ms=12, label="secondary center")
ax.set_aspect("equal")
ax.set_xlabel("x (SPAM units)")
ax.set_ylabel("y (SPAM units)")
ax.legend(markerscale=5)
plt.show()

## Seeds
`seed` sets where the disk particles start. The same seed gives exactly the same particles; a different seed
gives a different, equally valid set. With no seed, every run differs, as in Wallin's original. The
secondary's center has no random numbers, so it's the same in every case.

In [ ]:
same_seed = sim.simulate(params, config, seed=1)
other_seed = sim.simulate(params, config, seed=2)
print("seed 1 again, particles identical:", np.array_equal(same_seed, particles))
print("seed 2, particles match:          ", np.allclose(other_seed[:-1], particles[:-1]))
print("seed 2, secondary's center matches:", np.allclose(other_seed[-1], particles[-1]))

## The starting state
`sim.initial_state` takes the same arguments as `simulate`, and returns the particles before SPAM takes a step.
With the same seed, it's exactly the state `simulate` starts from. The secondary starts far away, before the
encounter.

In [ ]:
start = sim.initial_state(params, config, seed=1)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(start[:N1, 0], start[:N1, 1], s=1, label="primary disk")
ax.scatter(start[N1:N1 + N2, 0], start[N1:N1 + N2, 1], s=1, label="secondary disk")
ax.set_aspect("equal")
ax.set_title("start")
ax.legend(markerscale=5)
plt.show()

## Disk heat
`heat1` and `heat2` add random motion to each disk's particle velocities (0 by default). `heat_mode` sets how:

| `heat_mode` | Kick added to each of vx, vy, vz | Effect |
|---|---|---|
| `"centered"` (default) | between −heat/2 and +heat/2 | Random motion, no drift |
| `"wallin"` (the original) | between 0 and heat | The same spread, but always positive, so a heated disk also drifts toward +x, +y, +z by about heat/2 |

The kicks show up as the difference between a cold and a hot start with one seed:

In [ ]:
HEAT = 0.5
cold = sim.initial_state(params, config, seed=1)
for mode in ["centered", "wallin"]:
    hot = sim.initial_state(params, replace(config, heat1=HEAT, heat2=HEAT, heat_mode=mode), seed=1)
    kick = hot[:-1, 3:] - cold[:-1, 3:]
    print(f"{mode:8s}  kick range: {kick.min():+.3f} to {kick.max():+.3f}   mean per axis: {kick.mean(axis=0).round(3)}")

## The galaxy's mass model
Both galaxies are scaled copies of one **unit galaxy**: a halo, a disk and a bulge, stored as tables over
radius. The force on a particle is the table's force × `mass1` (or `mass2`) from the 22 parameters.
`sim.mass_profile(config)` returns those tables. One SPAM distance unit is about 15 kpc. What each value does,
in the code and physically, is in `docs/spam_physics.md`.

| Field | Default (Wallin's) | What it does |
|---|---|---|
| `lnl` | 0.001 | Strength of dynamical friction on the secondary |
| `r_scale` | 10 | The halo's core radius. It also sets the tables' radius grid, so it stretches every component |
| `m_halo`, `m_bulge` | 5.8, 0.3333 | Halo and bulge masses |
| `h_bulge` | 0.2 | Bulge scale length |
| `h_disk` | 1.0 | Disk scale length, which also sets the disk's mass |
| `normalization` | `"total"` | `"total"`: divide by the actual total, so the unit galaxy's mass is 1 and the values above only shape it. `"wallin"` (the original): divide by a fixed 7.13333, so changing a mass changes the total |

### The unit galaxy
Enclosed mass of each component, and the force a particle feels. The halo, disk and bulge tables end at r = 2,
so the mass stops growing there.

In [ ]:
profile = sim.mass_profile(config)
r = profile["radius"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name in ["halo", "disk", "bulge", "total"]:
    axes[0].plot(r, profile[name], label=name)
axes[0].set_xlim(0, 3)
axes[0].set_xlabel("radius (SPAM units)")
axes[0].set_ylabel("enclosed mass (galaxy mass = 1)")
axes[0].legend()
axes[1].plot(r, profile["force"], label="SPAM's force")
axes[1].plot(r[1:], profile["total"][1:] / r[1:] ** 2, "--", label="enclosed mass / r²")
axes[1].set_xlim(0, 10)
axes[1].set_yscale("log")
axes[1].set_xlabel("radius (SPAM units)")
axes[1].legend()
plt.show()
for name in ["halo", "disk", "bulge", "total"]:
    print(f"{name:6s} {profile[name][-1]:.5f}")
print("total with wallin normalization:", sim.mass_profile(replace(config, normalization="wallin"))["total"][-1].round(5))

The dashed line is plain gravity from the enclosed mass. SPAM's force matches it (within 0.2%) from r = 0.05 to
the table's end at r = 10. Inside r = 0.05 it's deliberately smoothed, so particles near the center don't get a
huge kick. Past r = 10 the table ends, and SPAM keeps using its last row, so the force stops falling.

### What each value does
Each row doubles (or halves) one value and shows the total enclosed mass, with `"total"` normalization so every
curve ends at 1. `lnl` isn't in the tables: it only sets dynamical friction (below).

In [ ]:
changes = {
    "m_halo": [2.9, 11.6], "m_bulge": [0.15, 0.6], "h_bulge": [0.1, 0.4], "h_disk": [0.5, 2.0], "r_scale": [5.0, 20.0],
}
base = sim.mass_profile(config)
fig, axes = plt.subplots(len(changes), 1, figsize=(12, 3 * len(changes)), sharex=True, sharey=True)
for ax, (name, values) in zip(axes, changes.items()):
    ax.plot(base["radius"], base["total"], "k", label=f"{getattr(config, name)} (default)")
    for value in values:
        p = sim.mass_profile(replace(config, **{name: value}))
        ax.plot(p["radius"], p["total"], label=str(value))
    ax.set_xlim(0, 2)
    ax.set_ylabel("enclosed mass")
    ax.set_title(name, loc="left")
    ax.legend(loc="lower right")
axes[-1].set_xlabel("radius (SPAM units)")
fig.tight_layout()
plt.show()

**`r_scale` is the odd one.** It's the halo's core radius, but the tables' radius grid is built from it too,
so it stretches every component, while SPAM keeps reading the tables at fixed distances. In force that shows up
as (10 / `r_scale`)² outside the core:

In [ ]:
base_force = sim.mass_profile(config)["force"]
row = np.searchsorted(r, 5.0)  # r = 5
for r_scale in [5.0, 10.0, 20.0]:
    force = sim.mass_profile(replace(config, r_scale=r_scale))["force"]
    print(f"r_scale {r_scale:5.1f}: force at r = 5 is {force[row] / base_force[row]:.3f} x the default"
          f"   (10 / r_scale)^2 = {(10 / r_scale) ** 2:.3f}")

### Effect on a run
Raising `m_halo` to 11.6: with `"wallin"` the galaxies get much heavier, so the encounter changes a lot. With
`"total"` they keep their masses, and only the distribution changes. Same seed in every panel.

In [ ]:
small = replace(config, n1=1500, n2=1500)
runs = {
    "total, defaults": small,
    "total, m_halo 11.6": replace(small, m_halo=11.6),
    "wallin, m_halo 11.6": replace(small, m_halo=11.6, normalization="wallin"),
}
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
for ax, (title, cfg) in zip(axes, runs.items()):
    final = sim.simulate(params, cfg, seed=1)
    ax.scatter(final[:cfg.n1, 0], final[:cfg.n1, 1], s=1)
    ax.scatter(final[cfg.n1:-1, 0], final[cfg.n1:-1, 1], s=1)
    ax.set_title(title)
    ax.set_aspect("equal")
plt.show()

### Dynamical friction (`lnl`)
Friction slows the secondary as it moves through the primary's halo, but the halo's density table ends at
r = 2, so it only acts when the secondary passes inside that. The SDSS model above never does. Line 3 of
JSPAM's `data.txt` does:

In [ ]:
close = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                  12.97415, 135.11902, 97.54035, 305.01225, 0.3, 0.3, 0, 0, 0, 0, 0, 0])
tiny = replace(config, n1=5, n2=5)
for lnl in [0.0, 0.001, 0.1, 0.5]:
    end = sim.simulate(close, replace(tiny, lnl=lnl), seed=1)[-1]
    print(f"lnl {lnl:5.3f}: secondary ends at {end[:3].round(4)}")